# Usecase: Bringing (almost) everything together

### Introduction
Using a certain file as a starting point, we want to enrich the data we have, and store it locally at first, and later in an Azure Blob Storage. We will use the techniques and libraries from the previous modules, and combine it in this single usecase.

### General overview
1. Retrieve data from 'movies.xml', and store it in a directory, based on your own design.
2. Use the data the access an [API](https://countrystatecity.in/docs/) and retrieve additional information on the countries. Store the information per country in separate JSON files.
3. Convert the JSON files to pickle objects, and send them to an Azure Blob Storage.
4. Create a logger that logs the entire process.

The usecase is quite flexible. You can make use of the libraries as you see fit, and use your own methods to come to a suitable end. The most important wish is that the we want the data to be enriched using this specific [API](https://countrystatecity.in/docs/), and that the enriched JSON files are stored as pickle files in a Azure Blob Storage container.

Enjoy!

In [ ]:
# Import section, all potentially necessary libraries

import json
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import os
import sys
import pathlib
import shutil
import pickle
import azure.storage.blob
import logging

---
## Step 1: Set up logging

Before we start processing data, we set up a logger that will track the entire pipeline. This way, every action is recorded both in the console and in a log file.

**Explanation:** We create a named logger with `logging.getLogger()` and configure two handlers: a `StreamHandler` for console output and a `FileHandler` to persist logs to disk. Both handlers use the same formatter that includes a timestamp, log level, and message. Setting `propagate = False` prevents duplicate messages from the root logger. The level is set to INFO so we capture all operational messages without verbose debug output.

In [ ]:
# Create a logger
logger = logging.getLogger(name='usecase_pipeline')
logger.setLevel(logging.INFO)
logger.propagate = False

# Remove any existing handlers (useful when re-running the notebook)
logger.handlers.clear()

# Define a common format
log_format = logging.Formatter('%(asctime)s - %(levelname)s - %(name)s - %(message)s')

# Console handler
stream_handler = logging.StreamHandler()
stream_handler.setFormatter(log_format)
logger.addHandler(stream_handler)

# File handler
file_handler = logging.FileHandler('usecase_pipeline.log')
file_handler.setFormatter(log_format)
logger.addHandler(file_handler)

logger.info('Logger initialized successfully.')

---
## Step 2: Parse movies.xml

The file `movies.xml` actually contains country data in XML format. We need to parse this file and extract the relevant attributes for each country.

**Explanation:** We use `ET.parse()` from the `xml.etree.ElementTree` module to load the XML file and then call `getroot()` to access the root `<countries>` element. For each `<country>` child element, we extract the text content (country name) and the attributes `code`, `handle`, `continent`, and `iso` using `.text` and `.attrib`. Each country is stored as a dictionary in a list, giving us a structured representation of the data.

In [ ]:
# Parse the XML file
tree = ET.parse('movies.xml')
root = tree.getroot()

logger.info(f'Parsed movies.xml - found {len(root)} country elements.')

# Extract country data into a list of dictionaries
countries = []

for country_elem in root:
    country_data = {
        'name': country_elem.text,
        'code': country_elem.attrib.get('code'),
        'handle': country_elem.attrib.get('handle'),
        'continent': country_elem.attrib.get('continent'),
        'iso': country_elem.attrib.get('iso')
    }
    countries.append(country_data)

logger.info(f'Extracted {len(countries)} countries from XML.')

# Show the first 5 countries as a quick check
for c in countries[:5]:
    print(c)

---
## Step 3: Store country data locally as JSON files

We create an output directory and save each country as an individual JSON file. This gives us a structured local data store that we can enrich in the next step.

**Explanation:** We use `pathlib.Path` to define the output directory and call `mkdir(exist_ok=True)` so the code works both on a first run and on subsequent runs without raising an error. For each country dictionary, we write it as a JSON file using `json.dump()` with `indent=4` for human-readable formatting. The filename is based on the country code to keep things unique and short. Every file creation is logged for traceability.

In [ ]:
# Create output directory for JSON files
output_dir = pathlib.Path('country_data')
output_dir.mkdir(exist_ok=True)

logger.info(f'Output directory created: {output_dir.resolve()}')

# Save each country as a separate JSON file
for country in countries:
    file_path = output_dir / f"{country['code']}.json"
    with open(file_path, mode='w', encoding='utf-8') as f:
        json.dump(country, f, indent=4, ensure_ascii=False)
    logger.info(f"Saved {country['name']} to {file_path}")

logger.info(f'All {len(countries)} countries saved as JSON files.')

---
## Step 4: Enrich country data via the CountryStateCity API

We use the [CountryStateCity API](https://countrystatecity.in/docs/) to fetch additional information for each country. The API uses the two-letter ISO code (which matches the `code` field from our XML). To avoid rate limits during the exercise, we only enrich a small subset of countries.

**Explanation:** The CountryStateCity API requires an API key passed via the `X-CSCAPI-KEY` header. We use `requests.get()` to call the endpoint for each country, passing the uppercase two-letter country code. If the response status code is 200, we merge the API data with our existing local data using dictionary unpacking (`{**local, **api_response}`), which combines both dictionaries with API data overwriting any duplicate keys. We save the enriched result back as a JSON file. To be respectful of rate limits, we limit the demonstration to 5 example countries.

In [ ]:
# API configuration
API_BASE_URL = 'https://api.countrystatecity.in/v1/countries'
API_KEY = '<your-api-key>'  # Replace with your actual API key

headers = {
    'X-CSCAPI-KEY': API_KEY
}

# Create a directory for enriched data
enriched_dir = pathlib.Path('country_data_enriched')
enriched_dir.mkdir(exist_ok=True)

# Select a subset of countries to enrich (to avoid rate limits)
sample_countries = countries[:5]
logger.info(f'Enriching {len(sample_countries)} countries via API.')

for country in sample_countries:
    iso2_code = country['code'].upper()
    url = f'{API_BASE_URL}/{iso2_code}'
    
    logger.info(f"Requesting API data for {country['name']} ({iso2_code})...")
    
    try:
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            api_data = response.json()
            # Merge local data with API data
            enriched_data = {**country, **api_data}
            
            # Save enriched data
            file_path = enriched_dir / f"{country['code']}_enriched.json"
            with open(file_path, mode='w', encoding='utf-8') as f:
                json.dump(enriched_data, f, indent=4, ensure_ascii=False)
            
            logger.info(f"Enriched data for {country['name']} saved to {file_path}")
        else:
            logger.warning(f"API returned status {response.status_code} for {country['name']} ({iso2_code})")
    
    except requests.exceptions.RequestException as e:
        logger.error(f"Request failed for {country['name']}: {e}")

logger.info('API enrichment step completed.')

---
## Step 5: Convert enriched JSON files to pickle

Now we convert each enriched JSON file into a pickle file. Pickle is a more efficient binary format for Python objects and will be used for the upload to Azure Blob Storage.

**Explanation:** We iterate over all `.json` files in the enriched directory, load each one with `json.load()`, and then serialize the resulting dictionary using `pickle.dump()` in binary write mode ('wb'). The pickle files are saved in a separate directory to keep the data organized. Each conversion is logged so we can track progress and troubleshoot any issues.

In [ ]:
# Create a directory for pickle files
pickle_dir = pathlib.Path('country_data_pickle')
pickle_dir.mkdir(exist_ok=True)

# Convert each enriched JSON file to pickle
json_files = list(enriched_dir.glob('*.json'))
logger.info(f'Found {len(json_files)} enriched JSON files to convert to pickle.')

for json_file in json_files:
    # Load JSON data
    with open(json_file, mode='r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Save as pickle
    pickle_file_path = pickle_dir / f"{json_file.stem}.pickle"
    with open(pickle_file_path, mode='wb') as f:
        pickle.dump(data, f)
    
    logger.info(f'Converted {json_file.name} -> {pickle_file_path.name}')

logger.info('All JSON files converted to pickle.')

---
## Step 6: Upload pickle files to Azure Blob Storage

Finally, we upload each pickle file to an Azure Blob Storage container. This simulates a real-world scenario where processed data is stored in cloud storage for downstream consumption.

**Explanation:** We use `BlobClient.from_connection_string()` from the `azure.storage.blob` library to create a client for each blob. The connection string and container name are placeholders that should be replaced with actual credentials. For each pickle file, we open it in binary read mode and call `upload_blob()` with `overwrite=True` to handle re-runs gracefully. Each upload is wrapped in a try/except block so that a failure for one file does not halt the entire pipeline.

In [ ]:
from azure.storage.blob import BlobClient

# Azure Blob Storage configuration (replace with your actual credentials)
CONNECTION_STRING = '<your-azure-connection-string>'
CONTAINER_NAME = '<your-container-name>'

# Upload each pickle file
pickle_files = list(pickle_dir.glob('*.pickle'))
logger.info(f'Uploading {len(pickle_files)} pickle files to Azure Blob Storage.')

for pickle_file in pickle_files:
    blob_name = pickle_file.name
    
    try:
        blob_client = BlobClient.from_connection_string(
            conn_str=CONNECTION_STRING,
            container_name=CONTAINER_NAME,
            blob_name=blob_name
        )
        
        with open(pickle_file, mode='rb') as data:
            blob_client.upload_blob(data, overwrite=True)
        
        logger.info(f'Uploaded {blob_name} to container {CONTAINER_NAME}.')
    
    except Exception as e:
        logger.error(f'Failed to upload {blob_name}: {e}')

logger.info('Azure Blob Storage upload step completed.')

---
## Summary

In this usecase, we combined techniques from multiple modules into a single end-to-end pipeline:

1. **XML Parsing (Module 3):** Parsed `movies.xml` to extract country data using `xml.etree.ElementTree`.
2. **File System Operations (Module 4):** Created directories and saved structured data as individual JSON files using `pathlib` and `json`.
3. **API Requests (Module 2):** Enriched country data by calling the CountryStateCity API with `requests`, using authentication headers.
4. **Pickle (Module 5):** Converted enriched JSON data into pickle format for efficient binary storage.
5. **Azure Blob Storage (Module 6):** Uploaded pickle files to cloud storage using the `azure.storage.blob` library.
6. **Logging (Module 7):** Tracked every step of the pipeline with a configured logger writing to both console and file.

This pipeline demonstrates a realistic data engineering workflow: extract data from a source, enrich it with external information, transform it into an efficient format, and store it in the cloud -- all while maintaining observability through logging.